# Notebook 07 - Sensitivity Analysis (Per-Layer + Per-Expert)

## Learning objective
Identify which parts of DeepSeek-Coder-V2-Lite-Instruct are most sensitive to quantization, so Notebook 08 can make principled mixed-precision decisions.

The goal is not just to compare whole-model FP16, Q8, and Q4 runs. We already did that in earlier notebooks. The goal here is to ask a more specific question: when quality changes after quantization, which internal components are responsible?

## Key concepts
- **Sensitivity**: how much evaluation quality changes when a specific component is quantized more aggressively while the rest of the model is kept at the reference precision.
- **Ablation**: an experiment where we intentionally change one component, or one small group of components, and measure the effect.
- **Per-layer sensitivity**: some transformer blocks may tolerate Q4 well, while others may cause large degradations when quantized.
- **Per-expert sensitivity**: in a Mixture-of-Experts model, routed experts may not be equally important or equally robust to quantization.
- **Sensitivity delta**: `reference_pass@1 - candidate_pass@1`. Larger positive values mean more quality loss from quantizing that component.
- **Heatmap interpretation**: darker or higher-valued cells should mean larger performance drop, which implies a more precision-sensitive component.

## Why this notebook exists
Whole-model quantization answers questions like:

- How much smaller is Q4 than FP16?
- How much faster is Q4 than FP16?
- How much does HumanEval pass@1 change when the entire model is quantized?

Sensitivity analysis asks a different question:

- If only this layer group is quantized, how much does quality change?
- If only this expert is quantized, how much does quality change?
- Which components should stay at higher precision in a mixed-precision model?

This is the bridge from benchmarking to understanding. Notebook 08 should use these results to decide which layers or experts can safely use lower precision and which should be protected.

## Reference and candidate runs
Each valid sensitivity experiment needs two things:

1. A **reference run**, where the model uses the chosen baseline precision.
2. A **candidate run**, where one target component is quantized differently and everything else is held constant.

For this notebook, the reference should be a backend-consistent FP16 run. We should avoid comparing a `transformers` FP16 run directly against a `llama.cpp` GGUF run unless the difference is explicitly part of the experiment.

## Critical implementation question
The central engineering question for this notebook is: how do we actually create a candidate model where only a chosen layer group or expert is quantized?

Claiming an ablation in a manifest is not enough. A valid ablation requires either:

- a real model artifact where the selected component has been quantized and the rest has not, or
- a runtime mechanism that swaps or quantizes the selected component before evaluation.

Until that mechanism exists, we can define the experimental design and evaluation harness, but we should not treat the notebook as having completed sensitivity analysis.

## Inputs from previous notebooks
- `results/01_model_exploration.json` (architecture details: number of layers, MoE layers, expert counts)
- `results/03_baseline_evaluation.json` (original FP16 HumanEval reference from the `transformers` path)
- `results/05_ptq_artifacts_summary.json` (quantized artifact metadata)
- `results/06_benchmarking_perf_snapshot.json` (cross-precision speed/memory context)
- Optional Notebook 07 parity results, if regenerated, for backend-consistent FP16/Q8/Q4 HumanEval comparisons

## Outputs this notebook should produce
- A clearly documented reference baseline for sensitivity deltas
- A first-pass ablation design for layer groups and expert samples
- At least one valid end-to-end ablation run before building sweep automation
- Layer sensitivity table and ranked list, once enough ablations exist
- Expert sensitivity table and ranked list, once enough ablations exist
- Layer/expert heatmaps saved to `results/`
- A recommended precision policy for Notebook 08

## Recommended workflow
1. Load architecture and prior benchmark context.
2. Choose the reference baseline and document why it is valid.
3. Define the sensitivity metric.
4. Design the first small ablation: one layer group or one expert sample.
5. Build or load the corresponding candidate model.
6. Run HumanEval on the candidate with the same harness as the reference.
7. Record the delta and inspect whether the result makes sense.
8. Only after one valid ablation works, generalize to a manifest-driven sweep.

## Important cleanup rule
Keep whole-model parity benchmarking separate from sensitivity analysis. Evaluating full `fp16`, `q8_0`, and `q4_k_m` artifacts is useful context, but it does not by itself tell us which layers or experts are sensitive.

In [4]:
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
from datasets import load_dataset
from torch import nn
import torch
import datetime as dt
import json
import os
import sys
import time
import gc
import math

# HumanEval executes generated code in forked subprocesses. Disable tokenizer
# worker parallelism up front to avoid non-blocking fork warnings.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [5]:
PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"

exploration_path = RESULTS_DIR / "01_model_exploration.json"
benchmark_path = RESULTS_DIR / "06_benchmarking_perf_snapshot.json"

with exploration_path.open() as f:
    exploration = json.load(f)

with benchmark_path.open() as f:
    benchmark = json.load(f)

benchmark_results = benchmark["benchmark_results"]
fp16_quality = benchmark_results["measurements"]["fp16"]["quality"]

print("Model:", exploration["model_id"])
print("Number of total layers:", exploration["num_layers"])
print("Number of MoE layers:", len(exploration["moe_layers"]))
print("Number of routed experts per MoE layer:", exploration["n_routed_experts"])
print("FP16 GGUF artifact size, GiB:", benchmark_results["artifacts"]["fp16"]["artifact_size_gib"])
print("Notebook 03 FP16 HumanEval pass@1:", fp16_quality["humaneval_pass_at_1_percent"])


Model: deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct
Number of total layers: 27
Number of MoE layers: 26
Number of routed experts per MoE layer: 64
FP16 GGUF artifact size, GiB: 29.27
Notebook 03 FP16 HumanEval pass@1: 60.36585365853659


## First ablation question

Notebook 04 already studied quantize/dequantize mechanics and measured logits drift. We will not repeat that analysis here.

Notebook 07 uses the same simulated quantization mechanism only as an intervention for ablation experiments. The task-level question is:

> If we perturb one selected component with simulated lower precision, does HumanEval pass@1 degrade relative to the same model without that perturbation?

We start with dense layer 0 because it is structurally simpler than the MoE layers. Later ablations can target MoE routed experts.

Measured effect:

`delta_pass@1 = reference_pass@1 - candidate_pass@1`


## Methodology

Notebook 06 used GGUF artifacts to measure deployment behavior: artifact size, memory use, and inference speed.

Notebook 07 uses runtime PyTorch ablations to estimate component-level sensitivity. This gives us direct control over individual tensors and modules, which is needed for per-layer and per-expert experiments.

These ablations are sensitivity estimates, not deployment benchmarks. Deployment behavior remains the role of the GGUF artifacts from Notebooks 05 and 06.


In [7]:
model_id = exploration["model_id"]
remote_code_revision = "refs/pr/10"

config = AutoConfig.from_pretrained(
    model_id,
    trust_remote_code=True,
    code_revision=remote_code_revision,
)

print("model_id:", model_id)
print("remote_code_revision:", remote_code_revision)
print(type(config))
print(config)


/Users/jarrett/dev/quantization-study/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model_id: deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct
remote_code_revision: refs/pr/10
<class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.configuration_deepseek.DeepseekV2Config'>
DeepseekV2Config {
  "_name_or_path": "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct",
  "architectures": [
    "DeepseekV2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct--configuration_deepseek.DeepseekV2Config",
    "AutoModel": "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct--modeling_deepseek.DeepseekV2Model",
    "AutoModelForCausalLM": "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct--modeling_deepseek.DeepseekV2ForCausalLM"
  },
  "aux_loss_alpha": 0.001,
  "bos_token_id": 100000,
  "eos_token_id": 100001,
  "ep_size": 1,
  "first_k_dense_replace": 1,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size"

## Runtime compatibility note

DeepSeek-Coder-V2-Lite-Instruct relies on Hugging Face remote model code. For reproducibility, this project pins `transformers==4.40.2` and uses DeepSeek's `refs/pr/10` remote-code revision, which fixes a Mac-specific optional `flash_attn` import issue.

We also force `attn_implementation="eager"`, matching DeepSeek's own V2 Transformers example and avoiding Flash Attention on this machine.


In [8]:
for attr in [
    "num_hidden_layers",
    "n_routed_experts",
    "num_experts_per_tok",
    "first_k_dense_replace",
    "moe_layer_freq",
    "hidden_size",
    "intermediate_size",
]:
    print(attr, getattr(config, attr, None))

num_hidden_layers 27
n_routed_experts 64
num_experts_per_tok 6
first_k_dense_replace 1
moe_layer_freq 1
hidden_size 2048
intermediate_size 10944


In [9]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Match the Notebook 03 baseline path while pinning the remote code revision
# that avoids the optional flash_attn import issue on Mac.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    code_revision=remote_code_revision,
    attn_implementation="eager",
)
model.eval()
model.config.use_cache = True
model.generation_config.use_cache = True

print("Tokenizer:", type(tokenizer))
print("Model:", type(model))
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)
print("attention implementation:", model.config._attn_implementation)
print("use_cache:", model.generation_config.use_cache)
print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/Users/jarrett/dev/quantization-study/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Tokenizer: <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>
Model: <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2ForCausalLM'>
Model device: cpu
Model dtype: torch.bfloat16
attention implementation: eager
use_cache: True
pad_token: <｜end▁of▁sentence｜>
eos_token: <｜end▁of▁sentence｜>


In [6]:
for name, module in model.named_modules():
    if "layers.0" in name:
        print(name, type(module))

model.layers.0 <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2DecoderLayer'>
model.layers.0.self_attn <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2Attention'>
model.layers.0.self_attn.q_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.kv_a_proj_with_mqa <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.kv_a_layernorm <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2RMSNorm'>
model.layers.0.self_attn.kv_b_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.o_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.rotary_emb <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.

Now that we've loaded the model, let's inspect its module tree in a targeted way.  We want to identify the exact names for layer 0 and the expert modules.

The first ablation will target a small, explicit set of weights.  We don't want to use vague labels like 'layer 0' until we know which tensors that includes.

In [10]:
print(type(model))
print(type(model.model))
print(len(model.model.layers))

<class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2ForCausalLM'>
<class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2Model'>
27


In [11]:
layer0 = model.model.layers[0]

for name, module in layer0.named_modules():
    print(f"{name:60s} {type(module)}")

                                                             <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2DecoderLayer'>
self_attn                                                    <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2Attention'>
self_attn.q_proj                                             <class 'torch.nn.modules.linear.Linear'>
self_attn.kv_a_proj_with_mqa                                 <class 'torch.nn.modules.linear.Linear'>
self_attn.kv_a_layernorm                                     <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2RMSNorm'>
self_attn.kv_b_proj                                          <class 'torch.nn.modules.linear.Linear'>
self_attn.o_proj                                

In [12]:
first_moe_layer_idx = exploration["moe_layers"][0]
moe_layer = model.model.layers[first_moe_layer_idx]

print("first_moe_layer_idx:", first_moe_layer_idx)

for name, module in moe_layer.named_modules():
    print(f"{name:80s} {type(module)}")

first_moe_layer_idx: 1
                                                                                 <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2DecoderLayer'>
self_attn                                                                        <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2Attention'>
self_attn.q_proj                                                                 <class 'torch.nn.modules.linear.Linear'>
self_attn.kv_a_proj_with_mqa                                                     <class 'torch.nn.modules.linear.Linear'>
self_attn.kv_a_layernorm                                                         <class 'transformers_modules.deepseek-ai.DeepSeek-Coder-V2-Lite-Instruct.432f2d153b34b8b1f52c8d4cbac22c77b2b1ddc2.modeling_deepseek.DeepseekV2RMSNorm'>
self_attn.kv_b_proj        

In [13]:
print("Layer 0 parameters:")
for name, param in layer0.named_parameters():
    print(f"{name:80s} {tuple(param.shape)} {param.dtype}")

print()
print("First MoE layer parameters:")
for name, param in moe_layer.named_parameters():
    print(f"{name:80s} {tuple(param.shape)} {param.dtype}")


Layer 0 parameters:
self_attn.q_proj.weight                                                          (3072, 2048) torch.bfloat16
self_attn.kv_a_proj_with_mqa.weight                                              (576, 2048) torch.bfloat16
self_attn.kv_a_layernorm.weight                                                  (512,) torch.bfloat16
self_attn.kv_b_proj.weight                                                       (4096, 512) torch.bfloat16
self_attn.o_proj.weight                                                          (2048, 2048) torch.bfloat16
mlp.gate_proj.weight                                                             (10944, 2048) torch.bfloat16
mlp.up_proj.weight                                                               (10944, 2048) torch.bfloat16
mlp.down_proj.weight                                                             (2048, 10944) torch.bfloat16
input_layernorm.weight                                                           (2048,) torch.bfloat16
post_atte

## First ablation target

Layer 0 is the only dense decoder layer. Layers 1-26 are MoE layers.

For the first ablation, we perturb only the `torch.nn.Linear` weights in layer 0:

- self-attention projections
- dense MLP projections

We do not modify layer norms, rotary embeddings, token embeddings, the LM head, MoE routers, routed experts, or shared experts.

This first ablation is intentionally simple. Its purpose is to validate the mechanics of targeted task-level sensitivity measurement before moving to MoE expert ablations.


In [14]:
def quantize_dequantize_per_channel(weight: torch.Tensor, n_bits: int = 8) -> torch.Tensor:
    """Simulate symmetric per-output-channel quantization for a Linear weight."""
    if weight.ndim != 2:
        raise ValueError(f"Expected a 2D Linear weight, got shape {tuple(weight.shape)}")
    if n_bits < 2:
        raise ValueError("n_bits must be >= 2")

    original_dtype = weight.dtype
    qmax = (2 ** (n_bits - 1)) - 1

    # Linear weights are [out_features, in_features]. Each output row gets its own scale.
    w = weight.detach().float()
    max_abs_per_row = w.abs().amax(dim=1, keepdim=True)
    scale = torch.where(
        max_abs_per_row > 0,
        max_abs_per_row / qmax,
        torch.ones_like(max_abs_per_row),
    )

    q = torch.clamp(torch.round(w / scale), -qmax, qmax)
    dequantized = q * scale
    return dequantized.to(dtype=original_dtype, device=weight.device)


def collect_linear_modules_in_layer(model, layer_idx: int):
    """Return all Linear modules inside one decoder layer."""
    layer = model.model.layers[layer_idx]
    modules = []

    for name, module in layer.named_modules():
        if isinstance(module, nn.Linear):
            full_name = f"model.layers.{layer_idx}.{name}"
            modules.append((full_name, module))

    return modules


layer0_linear_modules = collect_linear_modules_in_layer(model, layer_idx=0)

print(f"Found {len(layer0_linear_modules)} Linear modules in layer 0:")
print()
for name, module in layer0_linear_modules:
    print(f"{name:55s} {tuple(module.weight.shape)} {module.weight.dtype}")


Found 7 Linear modules in layer 0:

model.layers.0.self_attn.q_proj                         (3072, 2048) torch.bfloat16
model.layers.0.self_attn.kv_a_proj_with_mqa             (576, 2048) torch.bfloat16
model.layers.0.self_attn.kv_b_proj                      (4096, 512) torch.bfloat16
model.layers.0.self_attn.o_proj                         (2048, 2048) torch.bfloat16
model.layers.0.mlp.gate_proj                            (10944, 2048) torch.bfloat16
model.layers.0.mlp.up_proj                              (10944, 2048) torch.bfloat16
model.layers.0.mlp.down_proj                            (2048, 10944) torch.bfloat16


In [15]:
@torch.no_grad()
def apply_weight_ablation(modules, n_bits: int = 8):
    """Replace selected weights with simulated lower-precision versions."""
    originals = {}

    for name, module in modules:
        originals[name] = module.weight.detach().clone()
        module.weight.copy_(quantize_dequantize_per_channel(module.weight, n_bits=n_bits))

    return originals


@torch.no_grad()
def restore_weight_ablation(modules, originals):
    """Restore weights saved by apply_weight_ablation()."""
    for name, module in modules:
        if name not in originals:
            raise KeyError(f"Missing original weights for {name}")
        module.weight.copy_(originals[name])


In [16]:
def smoke_test_ablation_restore(modules, n_bits: int = 8):
    """
    Verify that an ablation changes at least one selected weight and then restores exactly.
    """
    if not modules:
        raise ValueError("No modules provided")

    target_name, target_module = modules[0]
    before = target_module.weight.detach().clone()

    originals = apply_weight_ablation(modules, n_bits=n_bits)
    after_ablation = target_module.weight.detach().clone()

    restore_weight_ablation(modules, originals)
    after_restore = target_module.weight.detach().clone()

    ablation_delta = (after_ablation - before).abs().max().item()
    restore_delta = (after_restore - before).abs().max().item()

    return {
        "target_module": target_name,
        "n_modules_ablated": len(modules),
        "n_bits": n_bits,
        "max_change_after_ablation": ablation_delta,
        "max_difference_after_restore": restore_delta,
    }


smoke_test_ablation_restore(layer0_linear_modules, n_bits=8)

{'target_module': 'model.layers.0.self_attn.q_proj',
 'n_modules_ablated': 7,
 'n_bits': 8,
 'max_change_after_ablation': 0.001953125,
 'max_difference_after_restore': 0.0}

This 5-problem run is only a pipeline smoke test. It verifies that we can run the same evaluator before and after a targeted weight ablation, restore the
model, and compute a pass@1 delta.

It should not be interpreted as a meaningful sensitivity score. Meaningful scores require a larger evaluation set, ideally the full 164-problem HumanEval
run.

In [17]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.humaneval_helpers import (
    clean_and_extract,
    execute_with_timeout,
    format_humaneval_prompt,
)

def generate_completion(problem, model, tokenizer, max_new_tokens=512):
    """Generate a HumanEval completion using the Notebook 03 generation setup."""
    formatted = format_humaneval_prompt(problem, tokenizer)
    inputs = tokenizer(formatted, return_tensors="pt")

    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1] :],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    generated = generated.replace("\r\n", "\n").replace("\r", "\n")
    generated = generated.replace("Ġ", " ").replace("Ċ", "\n").replace("č", "\n")
    return generated.rstrip()


def evaluate_problem(problem, model, tokenizer, timeout=10, max_new_tokens=512):
    generated = generate_completion(
        problem,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
    )
    code = clean_and_extract(generated, problem)
    full_code = code + "\n\n" + problem["test"] + f"\ncheck({problem['entry_point']})"
    result = execute_with_timeout(full_code, timeout=timeout)
    return {"task_id": problem["task_id"], "result": result, "generated": generated}


def run_humaneval_subset(model, tokenizer, max_problems: int = 5):
    """Run a HumanEval subset and return task-level results plus pass@1."""
    humaneval = load_dataset("openai/openai_humaneval", split="test")
    subset = humaneval.select(range(min(max_problems, len(humaneval))))

    results = []
    started = time.time()

    for i, problem in enumerate(subset, start=1):
        problem_started = time.time()
        result = evaluate_problem(problem, model=model, tokenizer=tokenizer)
        result["elapsed_sec"] = time.time() - problem_started
        results.append(result)

        n_passed = sum(1 for item in results if item["result"] == "passed")
        print(
            f"{result['task_id']}: {result['result']} "
            f"({result['elapsed_sec']:.1f}s) | running pass rate: {100.0 * n_passed / i:.2f}%"
        )

    n_total = len(results)
    n_passed = sum(1 for item in results if item["result"] == "passed")
    pass_at_1_percent = 100.0 * n_passed / n_total if n_total else 0.0

    return {
        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "n_total": n_total,
        "n_passed": n_passed,
        "pass_at_1_percent": pass_at_1_percent,
        "runtime_sec_total": time.time() - started,
        "results": results,
    }


In [18]:
smoke_result = smoke_test_ablation_restore(layer0_linear_modules, n_bits=8)
smoke_result


{'target_module': 'model.layers.0.self_attn.q_proj',
 'n_modules_ablated': 7,
 'n_bits': 8,
 'max_change_after_ablation': 0.001953125,
 'max_difference_after_restore': 0.0}

## Pipeline smoke test

The next cells run a 5-problem HumanEval smoke test before and after a targeted layer-0 ablation.

This verifies the evaluation loop, ablation application, restoration, and delta calculation. It is not a meaningful sensitivity score because 5 HumanEval problems are too few.


In [16]:
baseline_smoke = run_humaneval_subset(
    model=model,
    tokenizer=tokenizer,
    max_problems=5,
)

baseline_smoke["pass_at_1_percent"]


/Users/jarrett/dev/quantization-study/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/Users/jarrett/dev/quantization-study/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


HumanEval/0: passed (11.6s) | running pass rate: 100.00%
HumanEval/1: passed (25.5s) | running pass rate: 100.00%
HumanEval/2: passed (11.0s) | running pass rate: 100.00%
HumanEval/3: error: IndentationError: unindent does not match any outer indentation level (6.4s) | running pass rate: 75.00%
HumanEval/4: passed (8.8s) | running pass rate: 80.00%


80.0

In [17]:
originals = apply_weight_ablation(layer0_linear_modules, n_bits=8)

try:
    layer0_int8_smoke = run_humaneval_subset(
        model=model,
        tokenizer=tokenizer,
        max_problems=5,
    )
finally:
    restore_weight_ablation(layer0_linear_modules, originals)

layer0_int8_smoke["pass_at_1_percent"]


/Users/jarrett/dev/quantization-study/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/Users/jarrett/dev/quantization-study/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


HumanEval/0: passed (10.4s) | running pass rate: 100.00%
HumanEval/1: passed (24.1s) | running pass rate: 100.00%
HumanEval/2: passed (10.7s) | running pass rate: 100.00%
HumanEval/3: error: IndentationError: unindent does not match any outer indentation level (6.3s) | running pass rate: 75.00%
HumanEval/4: passed (8.1s) | running pass rate: 80.00%


80.0

In [18]:
smoke_delta = baseline_smoke["pass_at_1_percent"] - layer0_int8_smoke["pass_at_1_percent"]

print(f"Baseline smoke pass@1:   {baseline_smoke['pass_at_1_percent']:.2f}%")
print(f"Layer 0 INT8 pass@1:     {layer0_int8_smoke['pass_at_1_percent']:.2f}%")
print(f"Smoke sensitivity delta: {smoke_delta:.2f} percentage points")


Baseline smoke pass@1:   80.00%
Layer 0 INT8 pass@1:     80.00%
Smoke sensitivity delta: 0.00 percentage points


## First measured ablation

After the smoke test passes, run the first real task-level sensitivity measurement.

This cell uses 4-bit simulated per-channel quantization on all Linear weights in dense layer 0. It is more aggressive than INT8, so it is more likely to reveal whether this component is sensitive.

The full 164-problem run is the first result that should be saved and interpreted. It may take a while.


In [19]:
FULL_EVAL_PROBLEMS = 164

baseline_full = run_humaneval_subset(
    model=model,
    tokenizer=tokenizer,
    max_problems=FULL_EVAL_PROBLEMS,
)

baseline_full["pass_at_1_percent"]


HumanEval/0: passed (10.4s) | running pass rate: 100.00%
HumanEval/1: passed (24.3s) | running pass rate: 100.00%
HumanEval/2: passed (11.3s) | running pass rate: 100.00%
HumanEval/3: error: IndentationError: unindent does not match any outer indentation level (6.7s) | running pass rate: 75.00%
HumanEval/4: passed (8.2s) | running pass rate: 80.00%
HumanEval/5: passed (16.9s) | running pass rate: 83.33%
HumanEval/6: passed (14.6s) | running pass rate: 85.71%
HumanEval/7: passed (13.1s) | running pass rate: 87.50%
HumanEval/8: error: IndentationError: unexpected indent (8.2s) | running pass rate: 77.78%
HumanEval/9: passed (10.8s) | running pass rate: 80.00%
HumanEval/10: passed (21.4s) | running pass rate: 81.82%
HumanEval/11: error: IndentationError: unindent does not match any outer indentation level (7.8s) | running pass rate: 75.00%
HumanEval/12: passed (17.1s) | running pass rate: 76.92%
HumanEval/13: passed (10.8s) | running pass rate: 78.57%
HumanEval/14: error: IndentationError

70.1219512195122

In [20]:
originals = apply_weight_ablation(layer0_linear_modules, n_bits=4)

try:
    layer0_int4_full = run_humaneval_subset(
        model=model,
        tokenizer=tokenizer,
        max_problems=FULL_EVAL_PROBLEMS,
    )
finally:
    restore_weight_ablation(layer0_linear_modules, originals)

layer0_int4_full["pass_at_1_percent"]


HumanEval/0: passed (11.9s) | running pass rate: 100.00%
HumanEval/1: passed (27.4s) | running pass rate: 100.00%
HumanEval/2: passed (12.3s) | running pass rate: 100.00%
HumanEval/3: error: IndentationError: unindent does not match any outer indentation level (7.5s) | running pass rate: 75.00%
HumanEval/4: passed (9.5s) | running pass rate: 80.00%
HumanEval/5: passed (18.8s) | running pass rate: 83.33%
HumanEval/6: passed (15.5s) | running pass rate: 85.71%
HumanEval/7: passed (14.3s) | running pass rate: 87.50%
HumanEval/8: error: IndentationError: unexpected indent (9.1s) | running pass rate: 77.78%
HumanEval/9: passed (11.5s) | running pass rate: 80.00%
HumanEval/10: passed (24.4s) | running pass rate: 81.82%
HumanEval/11: passed (17.3s) | running pass rate: 83.33%
HumanEval/12: passed (19.3s) | running pass rate: 84.62%
HumanEval/13: passed (11.6s) | running pass rate: 85.71%
HumanEval/14: passed (10.7s) | running pass rate: 86.67%
HumanEval/15: passed (11.7s) | running pass rate:

71.34146341463415

In [21]:
layer0_int4_delta = baseline_full["pass_at_1_percent"] - layer0_int4_full["pass_at_1_percent"]
layer0_int4_relative_drop = (
    layer0_int4_delta / baseline_full["pass_at_1_percent"]
    if baseline_full["pass_at_1_percent"] else None
)

print(f"Baseline pass@1:      {baseline_full['pass_at_1_percent']:.2f}%")
print(f"Layer 0 INT4 pass@1:  {layer0_int4_full['pass_at_1_percent']:.2f}%")
print(f"Delta:                {layer0_int4_delta:.2f} percentage points")
print(f"Relative drop:        {100 * layer0_int4_relative_drop:.2f}%")


Baseline pass@1:      70.12%
Layer 0 INT4 pass@1:  71.34%
Delta:                -1.22 percentage points
Relative drop:        -1.74%


In [22]:
ablation_summary = {
    "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook": "07_sensitivity_analysis",
    "ablation_id": "layer_00_all_linear_int4_per_channel_simulated",
    "ablation_kind": "layer_all_linear",
    "target": {
        "layer_idx": 0,
        "modules": [name for name, _ in layer0_linear_modules],
        "n_modules": len(layer0_linear_modules),
    },
    "quantization": {
        "method": "simulated_symmetric_per_channel",
        "n_bits": 4,
        "runtime": "pytorch_float_weights_after_dequantization",
    },
    "evaluation": {
        "dataset": "openai/openai_humaneval",
        "n_total": baseline_full["n_total"],
        "metric": "pass_at_1_percent",
    },
    "reference": {
        "label": "current_fp_model_same_harness",
        "pass_at_1_percent": baseline_full["pass_at_1_percent"],
        "n_passed": baseline_full["n_passed"],
    },
    "candidate": {
        "label": "layer_00_all_linear_int4_per_channel_simulated",
        "pass_at_1_percent": layer0_int4_full["pass_at_1_percent"],
        "n_passed": layer0_int4_full["n_passed"],
    },
    "sensitivity": {
        "delta_pass_at_1_percent": layer0_int4_delta,
        "relative_drop": layer0_int4_relative_drop,
    },
    "notes": [
        "This is a component-level PyTorch simulated quantization ablation.",
        "It is not a GGUF deployment benchmark.",
        "The model weights were restored after candidate evaluation.",
    ],
}

out_path = RESULTS_DIR / "07_layer_00_all_linear_int4_ablation_summary.json"
with out_path.open("w") as f:
    json.dump(ablation_summary, f, indent=2)

print(f"Saved {out_path}")


Saved /Users/jarrett/dev/quantization-study/results/07_layer_00_all_linear_int4_ablation_summary.json


In [23]:
smoke_test_ablation_restore(layer0_linear_modules, n_bits=4)


{'target_module': 'model.layers.0.self_attn.q_proj',
 'n_modules_ablated': 7,
 'n_bits': 4,
 'max_change_after_ablation': 0.02685546875,
 'max_difference_after_restore': 0.0}

## Layer sensitivity screening

A full HumanEval run for every layer would be expensive. Instead, use a fixed smaller subset to screen all layers, then reserve full HumanEval for the most interesting candidates.

This screen is intentionally noisy. Its job is not to produce final claims; its job is to rank layers for follow-up.


In [25]:
SCREENING_PROBLEMS = 30
SCREENING_BITS = 4


def run_layer_ablation_screen(layer_idx: int, n_bits: int = SCREENING_BITS, max_problems: int = SCREENING_PROBLEMS):
    """Run a fixed-subset HumanEval screen for one layer ablation."""
    modules = collect_linear_modules_in_layer(model, layer_idx=layer_idx)
    originals = apply_weight_ablation(modules, n_bits=n_bits)

    try:
        candidate = run_humaneval_subset(
            model=model,
            tokenizer=tokenizer,
            max_problems=max_problems,
        )
    finally:
        restore_weight_ablation(modules, originals)

    return {
        "layer_idx": layer_idx,
        "n_bits": n_bits,
        "max_problems": max_problems,
        "n_modules": len(modules),
        "candidate_pass_at_1_percent": candidate["pass_at_1_percent"],
        "candidate_n_passed": candidate["n_passed"],
        "runtime_sec_total": candidate["runtime_sec_total"],
        "results": candidate["results"],
    }


def summarize_layer_screen(baseline, candidates):
    """Convert layer-screen candidate runs into a compact ranked table."""
    rows = []
    reference_pass = baseline["pass_at_1_percent"]

    for candidate in candidates:
        candidate_pass = candidate["candidate_pass_at_1_percent"]
        delta = reference_pass - candidate_pass
        rows.append(
            {
                "layer_idx": candidate["layer_idx"],
                "n_bits": candidate["n_bits"],
                "n_modules": candidate["n_modules"],
                "max_problems": candidate["max_problems"],
                "reference_pass_at_1_percent": reference_pass,
                "candidate_pass_at_1_percent": candidate_pass,
                "delta_pass_at_1_percent": delta,
                "candidate_n_passed": candidate["candidate_n_passed"],
                "runtime_sec_total": candidate["runtime_sec_total"],
            }
        )

    return sorted(rows, key=lambda row: row["delta_pass_at_1_percent"], reverse=True)


In [29]:
screening_baseline = run_humaneval_subset(
    model=model,
    tokenizer=tokenizer,
    max_problems=SCREENING_PROBLEMS,
)

screening_baseline["pass_at_1_percent"]


HumanEval/0: passed (12.0s) | running pass rate: 100.00%
HumanEval/1: passed (25.9s) | running pass rate: 100.00%
HumanEval/2: passed (11.2s) | running pass rate: 100.00%
HumanEval/3: error: IndentationError: unindent does not match any outer indentation level (6.8s) | running pass rate: 75.00%
HumanEval/4: passed (8.2s) | running pass rate: 80.00%
HumanEval/5: passed (17.2s) | running pass rate: 83.33%
HumanEval/6: passed (13.8s) | running pass rate: 85.71%
HumanEval/7: passed (13.8s) | running pass rate: 87.50%
HumanEval/8: error: IndentationError: unexpected indent (9.0s) | running pass rate: 77.78%
HumanEval/9: passed (11.2s) | running pass rate: 80.00%
HumanEval/10: passed (22.1s) | running pass rate: 81.82%
HumanEval/11: error: IndentationError: unindent does not match any outer indentation level (8.7s) | running pass rate: 75.00%
HumanEval/12: passed (18.5s) | running pass rate: 76.92%
HumanEval/13: passed (11.3s) | running pass rate: 78.57%
HumanEval/14: error: IndentationError

80.0

In [30]:
layer_screen_candidates = []

for layer_idx in range(exploration["num_layers"]):
    print(f"\n=== Layer {layer_idx} / {exploration['num_layers'] - 1} ===")
    layer_screen_candidates.append(
        run_layer_ablation_screen(
            layer_idx=layer_idx,
            n_bits=SCREENING_BITS,
            max_problems=SCREENING_PROBLEMS,
        )
    )

layer_screen_summary = summarize_layer_screen(screening_baseline, layer_screen_candidates)
layer_screen_summary[:5]



=== Layer 0 / 26 ===
HumanEval/0: passed (10.5s) | running pass rate: 100.00%
HumanEval/1: passed (24.6s) | running pass rate: 100.00%
HumanEval/2: passed (11.5s) | running pass rate: 100.00%
HumanEval/3: error: IndentationError: unindent does not match any outer indentation level (7.1s) | running pass rate: 75.00%
HumanEval/4: passed (8.0s) | running pass rate: 80.00%
HumanEval/5: passed (17.4s) | running pass rate: 83.33%
HumanEval/6: passed (13.6s) | running pass rate: 85.71%
HumanEval/7: passed (13.3s) | running pass rate: 87.50%
HumanEval/8: error: IndentationError: unexpected indent (8.7s) | running pass rate: 77.78%
HumanEval/9: passed (11.1s) | running pass rate: 80.00%
HumanEval/10: passed (21.9s) | running pass rate: 81.82%
HumanEval/11: passed (15.8s) | running pass rate: 83.33%
HumanEval/12: passed (18.0s) | running pass rate: 84.62%
HumanEval/13: passed (11.3s) | running pass rate: 85.71%
HumanEval/14: passed (9.4s) | running pass rate: 86.67%
HumanEval/15: passed (10.4s)

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 523a686b-c3ef-460f-af11-475536ad3804)')' thrown while requesting HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md
Retrying in 1s [Retry 1/5].


HumanEval/0: passed (62.8s) | running pass rate: 100.00%
HumanEval/1: passed (54.1s) | running pass rate: 100.00%
HumanEval/2: passed (59.2s) | running pass rate: 100.00%
HumanEval/3: error: IndentationError: unindent does not match any outer indentation level (48.6s) | running pass rate: 75.00%
HumanEval/4: passed (40.8s) | running pass rate: 80.00%
HumanEval/5: passed (61.0s) | running pass rate: 83.33%
HumanEval/6: passed (89.9s) | running pass rate: 85.71%
HumanEval/7: passed (63.5s) | running pass rate: 87.50%
HumanEval/8: error: IndentationError: unexpected indent (54.5s) | running pass rate: 77.78%
HumanEval/9: passed (102.5s) | running pass rate: 80.00%
HumanEval/10: passed (107.5s) | running pass rate: 81.82%
HumanEval/11: passed (84.2s) | running pass rate: 83.33%
HumanEval/12: passed (51.3s) | running pass rate: 84.62%
HumanEval/13: passed (59.5s) | running pass rate: 85.71%
HumanEval/14: error: IndentationError: unexpected indent (34.3s) | running pass rate: 80.00%
HumanEva

[{'layer_idx': 3,
  'n_bits': 4,
  'n_modules': 199,
  'max_problems': 30,
  'reference_pass_at_1_percent': 80.0,
  'candidate_pass_at_1_percent': 36.666666666666664,
  'delta_pass_at_1_percent': 43.333333333333336,
  'candidate_n_passed': 11,
  'runtime_sec_total': 328.8521957397461},
 {'layer_idx': 26,
  'n_bits': 4,
  'n_modules': 199,
  'max_problems': 30,
  'reference_pass_at_1_percent': 80.0,
  'candidate_pass_at_1_percent': 36.666666666666664,
  'delta_pass_at_1_percent': 43.333333333333336,
  'candidate_n_passed': 11,
  'runtime_sec_total': 327.48248171806335},
 {'layer_idx': 11,
  'n_bits': 4,
  'n_modules': 199,
  'max_problems': 30,
  'reference_pass_at_1_percent': 80.0,
  'candidate_pass_at_1_percent': 46.666666666666664,
  'delta_pass_at_1_percent': 33.333333333333336,
  'candidate_n_passed': 14,
  'runtime_sec_total': 304.8499598503113},
 {'layer_idx': 14,
  'n_bits': 4,
  'n_modules': 199,
  'max_problems': 30,
  'reference_pass_at_1_percent': 80.0,
  'candidate_pass_at_

In [31]:
layer_screen_path = RESULTS_DIR / "07_layer_int4_screening_summary.json"

layer_screen_output = {
    "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook": "07_sensitivity_analysis",
    "screening_kind": "all_layers_fixed_humaneval_subset",
    "n_bits": SCREENING_BITS,
    "max_problems": SCREENING_PROBLEMS,
    "reference": {
        "label": "current_fp_model_same_subset",
        "pass_at_1_percent": screening_baseline["pass_at_1_percent"],
        "n_passed": screening_baseline["n_passed"],
        "n_total": screening_baseline["n_total"],
    },
    "summary_ranked_by_delta_desc": layer_screen_summary,
    "candidate_runs": layer_screen_candidates,
    "notes": [
        "Positive delta means the layer ablation hurt pass@1 on the fixed screening subset.",
        "This is a noisy screening pass, not a final sensitivity claim.",
        "Use full HumanEval only on selected layers from this ranking.",
    ],
}

with layer_screen_path.open("w") as f:
    json.dump(layer_screen_output, f, indent=2)

print(f"Saved {layer_screen_path}")


Saved /Users/jarrett/dev/quantization-study/results/07_layer_int4_screening_summary.json


In [ ]:
# Do some memory cleanup before we proceed.
model_memory_cleanup()

## Full HumanEval follow-up for selected layers

The 30-problem screen is useful for ranking, but it is too small for final claims. Run full HumanEval only for a few selected layers: high-sensitivity candidates from the screen plus low-sensitivity controls.

Layer 0 already has a full result above, so the default follow-up list excludes it to avoid rerunning completed work.


In [20]:
FULL_FOLLOWUP_PROBLEMS = 164
FULL_FOLLOWUP_BITS = 4

# Chosen from the 30-problem screen:
# - layers 3 and 26 had the largest drops
# - layer 17 looked safe/noise-positive and acts as a control
FULL_FOLLOWUP_LAYER_IDXS = [3, 26, 17]


def run_layer_ablation_full(layer_idx: int, n_bits: int = FULL_FOLLOWUP_BITS):
    """Run full HumanEval for one selected layer ablation."""
    modules = collect_linear_modules_in_layer(model, layer_idx=layer_idx)
    originals = apply_weight_ablation(modules, n_bits=n_bits)

    try:
        candidate = run_humaneval_subset(
            model=model,
            tokenizer=tokenizer,
            max_problems=FULL_FOLLOWUP_PROBLEMS,
        )
    finally:
        restore_weight_ablation(modules, originals)

    return {
        "layer_idx": layer_idx,
        "n_bits": n_bits,
        "max_problems": FULL_FOLLOWUP_PROBLEMS,
        "n_modules": len(modules),
        "candidate_pass_at_1_percent": candidate["pass_at_1_percent"],
        "candidate_n_passed": candidate["n_passed"],
        "runtime_sec_total": candidate["runtime_sec_total"],
        "results": candidate["results"],
    }


In [21]:
layer_full_followup_candidates = []

for layer_idx in FULL_FOLLOWUP_LAYER_IDXS:
    print(f"\n=== Full follow-up: layer {layer_idx} ===")
    layer_full_followup_candidates.append(
        run_layer_ablation_full(
            layer_idx=layer_idx,
            n_bits=FULL_FOLLOWUP_BITS,
        )
    )

layer_full_followup_candidates



=== Full follow-up: layer 3 ===


/Users/jarrett/dev/quantization-study/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/Users/jarrett/dev/quantization-study/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


HumanEval/0: error: IndentationError: unindent does not match any outer indentation level (24.0s) | running pass rate: 0.00%
HumanEval/1: error: IndentationError: unexpected indent (13.7s) | running pass rate: 0.00%
HumanEval/2: error: SyntaxError: 'return' outside function (4.5s) | running pass rate: 0.00%
HumanEval/3: error: IndentationError: unindent does not match any outer indentation level (6.2s) | running pass rate: 0.00%
HumanEval/4: passed (7.9s) | running pass rate: 20.00%
HumanEval/5: passed (16.6s) | running pass rate: 33.33%
HumanEval/6: passed (14.2s) | running pass rate: 42.86%
HumanEval/7: passed (12.9s) | running pass rate: 50.00%
HumanEval/8: error: IndentationError: unindent does not match any outer indentation level (8.9s) | running pass rate: 44.44%
HumanEval/9: error: SyntaxError: 'return' outside function (8.1s) | running pass rate: 40.00%
HumanEval/10: passed (21.9s) | running pass rate: 45.45%
HumanEval/11: error: IndentationError: unindent does not match any o

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 5f5e26a8-aecb-4e86-955e-b9f360b2442d)')' thrown while requesting HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md
Retrying in 1s [Retry 1/5].


HumanEval/0: passed (18.9s) | running pass rate: 100.00%
HumanEval/1: passed (22.6s) | running pass rate: 100.00%
HumanEval/2: passed (11.3s) | running pass rate: 100.00%
HumanEval/3: error: IndentationError: unindent does not match any outer indentation level (6.3s) | running pass rate: 75.00%
HumanEval/4: passed (9.2s) | running pass rate: 80.00%
HumanEval/5: passed (16.6s) | running pass rate: 83.33%
HumanEval/6: passed (13.4s) | running pass rate: 85.71%
HumanEval/7: passed (13.4s) | running pass rate: 87.50%
HumanEval/8: error: IndentationError: unexpected indent (8.9s) | running pass rate: 77.78%
HumanEval/9: passed (20.1s) | running pass rate: 80.00%
HumanEval/10: passed (21.5s) | running pass rate: 81.82%
HumanEval/11: passed (16.1s) | running pass rate: 83.33%
HumanEval/12: passed (17.5s) | running pass rate: 84.62%
HumanEval/13: passed (10.3s) | running pass rate: 85.71%
HumanEval/14: passed (9.3s) | running pass rate: 86.67%
HumanEval/15: passed (10.8s) | running pass rate: 

[{'layer_idx': 3,
  'n_bits': 4,
  'max_problems': 164,
  'n_modules': 199,
  'candidate_pass_at_1_percent': 61.58536585365854,
  'candidate_n_passed': 101,
  'runtime_sec_total': 2142.45960021019,
  'results': [{'task_id': 'HumanEval/0',
    'result': 'error: IndentationError: unindent does not match any outer indentation level',
    'generated': ' ```python\n    for i in range(len(numbers)):\n        for j in range(i + 1, len(numbers)):\n            if abs(numbers[i] - numbers[j]) < threshold:\n                return True\n    return False\n```',
    'elapsed_sec': 24.045960187911987},
   {'task_id': 'HumanEval/1',
    'result': 'error: IndentationError: unexpected indent',
    'generated': " ```python\n    result = []\n    stack = []\n    current_group = []\n\n    for char in paren_string:\n        if char == ' ':\n            continue\n        if char == '(':\n            stack.append(char)\n            current_group.append(char)\n        elif char == ')':\n            stack.pop()\

In [23]:
layer_full_followup_runs_path = RESULTS_DIR / "07_selected_layer_int4_full_followup_runs.json"

layer_full_followup_runs = {
    "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook": "07_sensitivity_analysis",
    "run_kind": "selected_layers_full_humaneval_candidates",
    "n_bits": FULL_FOLLOWUP_BITS,
    "max_problems": FULL_FOLLOWUP_PROBLEMS,
    "selected_layers": FULL_FOLLOWUP_LAYER_IDXS,
    "candidate_runs": layer_full_followup_candidates,
    "notes": ["Each candidate run uses one in-place simulated INT4 layer ablation.",
        "Weights are restored after each candidate run.",
        "This file stores candidate runs only; compare against a separately saved full baseline run.",
    ],
}

with layer_full_followup_runs_path.open("w") as f:
    json.dump(layer_full_followup_runs, f, indent=2)

print(f"Saved {layer_full_followup_runs_path}")

Saved /Users/jarrett/dev/quantization-study/results/07_selected_layer_int4_full_followup_runs.json


In [22]:
layer_full_followup_summary = []
full_reference_pass = baseline_full["pass_at_1_percent"]

# Include the already-completed layer-0 full result as a control.
if "layer0_int4_full" in globals():
    layer_full_followup_summary.append(
        {
            "layer_idx": 0,
            "n_bits": 4,
            "n_modules": len(layer0_linear_modules),
            "max_problems": baseline_full["n_total"],
            "reference_pass_at_1_percent": full_reference_pass,
            "candidate_pass_at_1_percent": layer0_int4_full["pass_at_1_percent"],
            "delta_pass_at_1_percent": full_reference_pass - layer0_int4_full["pass_at_1_percent"],
            "candidate_n_passed": layer0_int4_full["n_passed"],
            "source": "existing_layer0_full_run",
        }
    )

for candidate in layer_full_followup_candidates:
    candidate_pass = candidate["candidate_pass_at_1_percent"]
    layer_full_followup_summary.append(
        {
            "layer_idx": candidate["layer_idx"],
            "n_bits": candidate["n_bits"],
            "n_modules": candidate["n_modules"],
            "max_problems": candidate["max_problems"],
            "reference_pass_at_1_percent": full_reference_pass,
            "candidate_pass_at_1_percent": candidate_pass,
            "delta_pass_at_1_percent": full_reference_pass - candidate_pass,
            "candidate_n_passed": candidate["candidate_n_passed"],
            "runtime_sec_total": candidate["runtime_sec_total"],
            "source": "full_followup_run",
        }
    )

layer_full_followup_summary = sorted(
    layer_full_followup_summary,
    key=lambda row: row["delta_pass_at_1_percent"],
    reverse=True,
)

layer_full_followup_summary


NameError: name 'baseline_full' is not defined

In [ ]:
layer_full_followup_path = RESULTS_DIR / "07_selected_layer_int4_full_followup_summary.json"

layer_full_followup_output = {
    "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook": "07_sensitivity_analysis",
    "followup_kind": "selected_layers_full_humaneval",
    "n_bits": FULL_FOLLOWUP_BITS,
    "max_problems": FULL_FOLLOWUP_PROBLEMS,
    "selected_layers": FULL_FOLLOWUP_LAYER_IDXS,
    "reference": {
        "label": "current_fp_model_full_humaneval",
        "pass_at_1_percent": baseline_full["pass_at_1_percent"],
        "n_passed": baseline_full["n_passed"],
        "n_total": baseline_full["n_total"],
    },
    "summary_ranked_by_delta_desc": layer_full_followup_summary,
    "candidate_runs": layer_full_followup_candidates,
    "notes": [
        "Positive delta means the selected layer ablation hurt full HumanEval pass@1.",
        "Layer 0 is included from the existing full run if layer0_int4_full is available.",
        "These runs validate whether the 30-problem screening ranking generalizes to the full benchmark.",
    ],
}

with layer_full_followup_path.open("w") as f:
    json.dump(layer_full_followup_output, f, indent=2)

print(f"Saved {layer_full_followup_path}")


## Memory cleanup

Run the next function when you are done with the current model instance or before loading a different model/runtime.


In [19]:
def model_memory_cleanup():
    large_names = [
        "model",
        "tokenizer",
        "config",
        "baseline_smoke",
        "layer0_int8_smoke",
        "baseline_full",
        "layer0_int4_full",
        "originals",
    ]

    for name in large_names:
        if name in globals():
            del globals()[name]

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except AttributeError:
            pass

    print("Deleted large notebook objects and requested Python/accelerator memory cleanup.")

## Checkpoint

Let's review the methodology and examine what we just learned.

We implemented int4 quantization on each layer of the model, but not all at once.  We quantized one layer, ran 30 HumanEval problems, reverted the layer to its original state, then quantized the next layer, and run 30 HumanEval problems again.  We repeated this process until every layer had been tested in this way.

This is not a direct 1:1 comparison to the full HumanEval test, which contains 164 problems.  The reason we only ran 30 problems is due to hardware limitations; it would have taken 2-3 days to run the full process on a consumer laptop.

This 30-question summary is enough to give us a coarse overview of which layers are more sensitive than others.


In [27]:
screening_path = RESULTS_DIR / "07_layer_int4_screening_summary.json"

with screening_path.open() as f:
    screening = json.load(f)

for layer in screening["summary_ranked_by_delta_desc"]:
    print(
        "Layer: ", layer["layer_idx"],
        "    ",
        layer["candidate_n_passed"],
        "/",
        layer["max_problems"],
        "=",
        math.floor(layer["candidate_pass_at_1_percent"]*10)/10,
        "%    delta ",
        math.floor(layer["delta_pass_at_1_percent"]*10)/10
    )

Layer:  3      11 / 30 = 36.6 %    delta  43.3
Layer:  26      11 / 30 = 36.6 %    delta  43.3
Layer:  11      14 / 30 = 46.6 %    delta  33.3
Layer:  14      17 / 30 = 56.6 %    delta  23.3
Layer:  1      18 / 30 = 60.0 %    delta  20.0
Layer:  8      18 / 30 = 60.0 %    delta  20.0
Layer:  10      18 / 30 = 60.0 %    delta  20.0
Layer:  6      19 / 30 = 63.3 %    delta  16.6
Layer:  4      21 / 30 = 70.0 %    delta  10.0
Layer:  7      21 / 30 = 70.0 %    delta  10.0
Layer:  12      22 / 30 = 73.3 %    delta  6.6
Layer:  24      22 / 30 = 73.3 %    delta  6.6
Layer:  13      23 / 30 = 76.6 %    delta  3.3
Layer:  18      23 / 30 = 76.6 %    delta  3.3
Layer:  22      23 / 30 = 76.6 %    delta  3.3
Layer:  25      23 / 30 = 76.6 %    delta  3.3
Layer:  9      24 / 30 = 80.0 %    delta  0.0
Layer:  5      25 / 30 = 83.3 %    delta  -3.4
Layer:  16      25 / 30 = 83.3 %    delta  -3.4
Layer:  0      26 / 30 = 86.6 %    delta  -6.7
Layer:  2      26 / 30 = 86.6 %    delta  -6.7
Layer:  1

#### Review
This list is ranked from highest to lowest sensitivity.  Put simply, Layer 3 had the largest performance degradation from int4 quantization, and Layer 17 had the least.

One limitation of our approach is that HumanEval is designed to with easier problems at the beginning of the list, and harder problems at the end of the list.  This 30-problem snapshot thus biases the results upward - you will see in a moment that some of the layers scored higher here than they did in Notebook 03.  This is more likely to be due to methodological differences than a true performance increase from quantization.

#### Summary
Sensitivity is highly non-uniform.  Some MoE blocks tolerate simulated int4 on this fixed subset of HumanEval problems, while layers 3, 26, 11, and 14 show large pass@1 drops and should be protected or retested with full HumanEval before any mixed-precision policy assigns them low protection.

#### Next Steps
Now that we know which layers are most and least sensitive to quantization, we'll run the full HumanEval for a small set:

High-sensitivity candidates: layers 3, 26

Low-sensitivity controls: layers 17, 0